In [5]:
import os
import numpy as np
import pandas as pd

In [6]:
healthy_dir = r"D:\M143020071\926\raw_data_result\iSKNA_signal\Sr10000_500_1000 envelope_10HZ\Non MI 200人"
patient_dir = r"D:\M143020071\926\raw_data_result\iSKNA_signal\Sr10000_500_1000 envelope_10HZ\MI 421人"

save_dir = r"D:\M143020071\926\Xgboost_result\特徵"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, 'iSKNA_420+200.npz')

In [7]:
def load_folder(folder_path): # 拔掉原本的 start_group_id
    data_list = []
    group_list = []

    files = sorted(os.listdir(folder_path))

    for file_name in files:
        if file_name.endswith(('.csv', '.txt', '.npy')) :
            file_path = os.path.join(folder_path, file_name)

            try:
                if file_name.endswith('.npy'):
                    data = np.load(file_path)
                elif file_name.endswith('.csv'):
                    data = pd.read_csv(file_path, header=None).values.flatten()
                elif file_name.endswith('.txt'):
                    data = np.loadtxt(file_path)

                data = np.array(data).flatten()

                if len(data) != 3001:
                    print(f'Skip {file_name}, len={len(data)}')
                    continue

                data_list.append(data)
                
                # 【修改這裡】：直接拿檔名當 ID。例如 '0001.npy' -> '0001'
                # 如果你的檔名是字串，group_list 裡面就會存字串，這對 GroupKFold 來說完全沒問題
                subject_id = os.path.splitext(file_name)[0] 
                group_list.append(subject_id)

            except Exception as e:
                print(f'Error: {file_name}')
                print(e)

    return np.array(data_list), np.array(group_list)

In [8]:
healthy_data, healthy_groups = load_folder(healthy_dir)
patient_data, patient_groups = load_folder(patient_dir)

all_data = np.concatenate([healthy_data, patient_data], axis=0)
groups = np.concatenate([healthy_groups, patient_groups]) # 裡面會是 ['0001', '0002', ...]

y = all_data[:, 0].astype(np.int8)  
X = all_data[:, 1:].astype(np.float32) 

np.savez(save_path, X=X, y=y, groups=groups)

print("="*40)
print(f"健康組資料形狀: {healthy_data.shape}，病人組資料形狀: {patient_data.shape}")
print(f"合併後 X 矩陣形狀: {X.shape}")
print(f"y 標籤內包含的不重複值: {np.unique(y)}") 
print(f"y 標籤中 0 的數量: {np.sum(y == 0)}，1 的數量: {np.sum(y == 1)}")
print("="*40)

# 確認有 0 也有 1 再存檔
if len(np.unique(y)) < 2:
    print("❌ 警告：y 裡面只有單一種標籤，請先不要跑模型，檢查兩組訊號的第一個點！")
else:
    np.savez(save_path, X=X, y=y, groups=groups)
    print(f'儲存成功！檔案已寫入:\n{save_path}')

健康組資料形狀: (200, 3001)，病人組資料形狀: (420, 3001)
合併後 X 矩陣形狀: (620, 3000)
y 標籤內包含的不重複值: [0 1]
y 標籤中 0 的數量: 200，1 的數量: 420
儲存成功！檔案已寫入:
D:\M143020071\926\Xgboost_result\特徵\iSKNA_420+200.npz
